# Feature selection — SageMaker Processing

Run the first-stage, four-level feature selection for the configured MLTools, Thoth, and Customizados books. The notebook only supplies Studio credentials and starts or resumes the orchestrator. A run-scoped Athena sample and S3 manifest are created by `fs_orchestrator.py`.

Place `fs_orchestrator.py` and `fs_processor.py` next to this notebook. Set `FS_RESUME_RUN_ID` to resume a run from its S3 manifest. The deployment environment uses the Scikit-learn Processing image `1.2-1`.


## Studio connections and configuration


In [ ]:
import os
from pathlib import Path

import boto3
import sagemaker
from sagemaker.network import NetworkConfig
from MLTools import Connections as mlcon
from global_info_studio import GlobalInfoStudio
from fs_orchestrator import default_config, run_pipeline, FRAMEWORK_VERSION

region = 'sa-east-1'
info = GlobalInfoStudio().get_global_info()
role = info['role']
bucket = info['default_bucket']
network_config = NetworkConfig(
    enable_network_isolation=False,
    encrypt_inter_container_traffic=True,
    security_group_ids=[info['sg_default']],
    subnets=[info['subnet_id']],
)
boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session, default_bucket=bucket)
athena = mlcon.Athena()
config = default_config(
    bucket,
    workgroup=os.getenv('FS_ATHENA_WORKGROUP', 'analytics-workgroup'),
    athena_output=os.getenv('FS_ATHENA_OUTPUT', f's3://{bucket}/athena-results/'),
)
resume_run_id = os.getenv('FS_RESUME_RUN_ID') or None
script_path = Path('fs_processor.py').resolve()
if not script_path.is_file():
    raise FileNotFoundError(f'Place fs_processor.py next to the notebook: {script_path}')
print('Region:', region, '| Processing image:', FRAMEWORK_VERSION)
print('Bucket:', bucket, '| books: MLTools, Thoth, Customizados')
print('Resume run:', resume_run_id or 'new run')


## Plan and execute the run

The orchestrator validates sample and book joins, profiles each book, chooses batch size and Processing instance, and maintains at most 10 active jobs. Results are written under `feature_selection_saidas/<run_id>/` and a matching S3 prefix. Batch-local correlation screening does not compare feature pairs in different batches.


In [ ]:
summary = run_pipeline(
    athena=athena,
    glue=boto_session.client('glue'),
    s3=boto_session.client('s3'),
    sm=boto_session.client('sagemaker'),
    bucket=bucket,
    role=role,
    session=sagemaker_session,
    network_config=network_config,
    code_path=str(script_path),
    output_dir='feature_selection_saidas',
    config=config,
    resume_run_id=resume_run_id,
)
print('Run:', summary['run_id'])
print('Sample rows:', summary['sample_count'])
print('Book statuses:', summary['book_status'])
print('Selected feature rows:', summary['selected_feature_rows'])
